# KadiPy Market - Module Forecasting

Ce notebook présente les prévisions de prix basées sur les modèles légers embarqués dans KadiPy.

---

## 1. Initialisation du Module de Prévision

Pour la version 0.1.0, au lieu d'utiliser des bibliothèques lourdes comme TensorFlow ou PyTorch, nous avons implémenté des alternatives extrêmement légères et rapides basées sur `scikit-learn` et `scipy`.
- **Régression avec termes de Fourier** (remplace Prophet pour la saisonnalité)
- **Perceptron Multicouche (MLPRegressor)** (remplace LSTM pour les séries non linéaires)
- **Fenêtres glissantes de variance** (remplace GARCH pour la volatilité)

**Pourquoi ?** L'objectif est d'avoir un package léger, rapide à installer sur n'importe quel serveur, et qui s'exécute avec peu de ressources.

In [2]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

from kadi.market.forecasting import MarketForecasting

forecaster = MarketForecasting()
print("Module Forecasting initialisé avec succès !")

Module Forecasting initialisé avec succès !


---
## 2. Prévision de prix à court terme (`days_ahead=14`)

La méthode `predict_price` est le cœur de ce module. Elle prend le nom de la culture (`crop`), le marché cible (`market`) et l'horizon de prévision (`days_ahead`).

**Explication de la sortie :**
La fonction ne retourne pas qu'un simple prix. Elle renvoie un dictionnaire complet :
- `predicted_price` : L'estimation la plus probable du prix.
- `low_90` et `high_90` : L'intervalle de confiance à 90%. Cela signifie que le système est confiant à 90% que le vrai prix se situera entre ces deux bornes.
- `model_used` : Le modèle interne qui a été sélectionné pour cette prédiction.

In [3]:
prevision = forecaster.predict_price(crop='maize', market='cotonou', days_ahead=14, confidence_interval=0.9)

print("Résultat de la prévision à 14 jours :")
for key, value in prevision.items():
    print(f"- {key} : {value}")

Résultat de la prévision à 14 jours :
- predicted_price : 342.0
- low_90 : 236.75
- high_90 : 447.25
- confidence : 0.9
- model_used : ensemble (ARIMA, Prophet-like, GARCH-like)
- rmse : 51.19


---
## 3. Comprendre la modélisation de la volatilité

En économie agricole, plus on essaie de prédire loin dans le futur, plus l'incertitude est grande. Notre alternative au modèle GARCH simule ce comportement : la variance augmente avec l'horizon temporel.

**Explication de la sortie :**
Regardez les intervalles affichés ci-dessous. 
- À 30 jours, l'écart entre le minimum et le maximum (l'intervalle de confiance) est modéré.
- À 90 jours, cet écart devient beaucoup plus grand. Le système avoue mathématiquement qu'il y a plus de risques et d'incertitudes sur une prévision à 3 mois qu'à 1 mois.

In [4]:
print("Prévision à 30 jours :")
prev_30 = forecaster.predict_price(crop='maize', market='cotonou', days_ahead=30)
print(f"Prix: {prev_30['predicted_price']:.2f} XOF/kg, Intervalle: [{prev_30['low_90']:.2f}, {prev_30['high_90']:.2f}]")

print("\nPrévision à 90 jours :")
prev_90 = forecaster.predict_price(crop='maize', market='cotonou', days_ahead=90)
print(f"Prix: {prev_90['predicted_price']:.2f} XOF/kg, Intervalle: [{prev_90['low_90']:.2f}, {prev_90['high_90']:.2f}]")

Prévision à 30 jours :
Prix: 390.00 XOF/kg, Intervalle: [214.30, 565.70]

Prévision à 90 jours :
Prix: 570.00 XOF/kg, Intervalle: [125.23, 1014.77]
